# Hub-and-Spoke Multi-Agent Memory Evaluation

## The Problem

In a multi-agent system, the most important failures happen *before* the final answer is produced. A sub-agent may receive stale context, the coordinator may update the budget but not re-dispatch the hotel search, or two agents may work from different versions of the same plan. These are **memory and coordination failures** — and they're invisible unless you instrument for them.

## What This Notebook Does

We run a **travel planning** hub-and-spoke system through two sessions:

1. **Simple Session** — User requests a trip with a fixed budget. No changes mid-conversation. This is the baseline.
2. **Session with Feedback** — Same initial request, but the user changes the budget mid-session. This tests whether agents pick up the update, whether the coordinator re-dispatches correctly, and whether the final itinerary reflects the new constraints.

The metrics tell us what happened — no manual labeling of "good" or "bad" needed.

## Architecture

```


                    ┌─────────────────────┐
                    │   User / Customer    │
                    └──────────┬──────────┘
                               │
                    ┌──────────▼──────────┐
                    │   Travel Coordinator │  <-------- Hub (no memory)
                    │   Decides which      │    Routes queries to spokes
                    │   spoke to call      │    Compresses user message
                    └──┬───────┬────────┬──┘    into handoff query
                       │       │        │
              ┌────────▼──┐ ┌──▼─────┐ ┌▼──────────┐
              │  Flight   │ │ Hotel  │ │ Itinerary │
              │  Agent    │ │ Agent  │ │ Agent     │
              └─────┬─────┘ └───┬────┘ └─────┬─────┘
                    │           │             │
              ┌─────▼───────────▼─────────────▼─────┐
              │     AgentCore Shared Memory          │
              │     (session_id shared,              │
              │      actor_id per agent)             │
              └─────────────────────────────────────┘





```

**Flow:**
- Coordinator receives user request and delegates via tool calls
- Each spoke reads shared memory (`get_last_k_turns`), processes the query, writes back (`create_event`)
- Itinerary agent runs last — consumes Flight + Hotel outputs from memory
- All context flows through the coordinator or through shared memory

## Metrics Evaluated

**Memory Context Metrics** (LLM-as-judge, 1-5 scale):

| Metric | What it measures |
|--------|------------------|
| Context Freshness | Is the agent working with the latest information? |
| Handoff Completeness | Did the coordinator include all facts the spoke needs? |
| Context Utilization | Did the spoke use the context it read from memory? |
| State Consistency | Do all spokes agree on key facts (budget, dates, etc.)? |
| Memory Write Accuracy | Is what the agent wrote to memory factually correct? |
| Redundant Context | How much repeated/irrelevant context was transferred? |
| CCR | Length ratio of handoff vs original (pure math) |

**Memory Latency Metrics** (timers + token counts):

| Metric | What it measures |
|--------|------------------|
| Memory Read/Write Latency | Time for memory operations |
| Coordination Latency % | Fraction of time spent on coordination vs reasoning |
| Coordination Token % | Fraction of tokens spent on context vs generation |

## Step 1 — Setup

In [ ]:
!pip install -qr requirements.txt

In [ ]:
import os
import time
import logging
from datetime import datetime
from IPython.display import display, Markdown

from strands import Agent, tool
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory import MemoryClient
from botocore.exceptions import ClientError

from metrics_collector import MetricsCollector
from model_config import (
    AGENT_MODEL_ID, FLIGHT_PROMPT, HOTEL_PROMPT,
    ITINERARY_PROMPT, HUB_PROMPT,
)

region = os.getenv("AWS_REGION", "us-west-2")

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s",
                    datefmt="%Y-%m-%d %H:%M:%S")
logger = logging.getLogger("hub-spoke-metrics")

## Step 2 — Create or Reuse AgentCore Memory

The memory resource is a long-lived container. Each session uses a fresh `session_id` so agents start with a clean slate — no data leaks between runs.

In [ ]:
memory_client = MemoryClient(region_name=region)
memory_name = "TravelMetrics_STM"
memory_id = None

try:
    print("Creating Memory (takes ~3 min first time)...")
    memory = memory_client.create_memory_and_wait(
        name=memory_name, description="Travel Metrics STM",
        strategies=[], max_wait=300, poll_interval=10)
    memory_id = memory["id"]
    print(f"Memory ready: {memory_id}")
except ClientError as e:
    if "already exists" in str(e):
        memories = memory_client.list_memories()
        memory_id = next(
            (m["id"] for m in memories
             if m["id"].startswith(memory_name) and m.get("status") == "ACTIVE"),
            None)
        print(f"Reusing existing memory: {memory_id}")
    else:
        raise

## Step 3 — Instrumented Memory Hook

Extends the standard `ShortTermMemoryHook` with:
- Latency timers around `get_last_k_turns` and `create_event`
- Metrics recording in `finally` blocks (never skipped even if memory fails)

In [ ]:
class CustomInstrumentationHook(HookProvider):

    def __init__(self, mem_client, mem_id, collector, agent_name):
        self.mem_client = mem_client
        self.mem_id = mem_id
        self.collector = collector
        self.agent_name = agent_name

    def on_agent_initialized(self, event: AgentInitializedEvent):
        context = ""
        try:
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")
            if not actor_id or not session_id:
                return

            t0 = time.perf_counter()
            recent = self.mem_client.get_last_k_turns(
                memory_id=self.mem_id, actor_id=actor_id,
                session_id=session_id, k=5, branch_name="main")
            self.collector.record_memory_read_latency(self.agent_name, time.perf_counter() - t0)

            if recent:
                parts = []
                for turn in recent:
                    for msg in turn:
                        parts.append(f"{msg['role'].title()}: {msg['content']['text']}")
                context = "\n".join(parts)
                event.agent.system_prompt += (
                    f"\n\nRecent conversation history:\n{context}"
                    "\n\nContinue naturally based on this context.")
                logger.info(f"[{self.agent_name}] Loaded {len(recent)} turns")
        except Exception as e:
            logger.error(f"[{self.agent_name}] Memory read failed: {e}")
        finally:
            self.collector.record_retrieved_context(self.agent_name, context)

    def on_message_added(self, event: MessageAddedEvent):
        last = event.agent.messages[-1]
        if last["role"] == "assistant":
            self.collector.record_response(self.agent_name, last["content"][0]["text"])
        try:
            actor_id = event.agent.state.get("actor_id")
            session_id = event.agent.state.get("session_id")
            if not actor_id or not session_id:
                return
            # Prefix content with agent name so other agents see who wrote it
            content = f"[{self.agent_name}] {last['content'][0]['text']}"
            t0 = time.perf_counter()
            self.mem_client.create_event(
                memory_id=self.mem_id, actor_id=actor_id,
                session_id=session_id,
                messages=[(content, last["role"])],
                metadata={"agent_name": {"stringValue": self.agent_name}})
            self.collector.record_memory_write_latency(self.agent_name, time.perf_counter() - t0)
        except Exception as e:
            logger.error(f"[{self.agent_name}] Memory write failed: {e}")

    def register_hooks(self, registry: HookRegistry):
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)
        registry.add_callback(MessageAddedEvent, self.on_message_added)

## Step 4 — Session Runner

Three spokes + one hub coordinator. `run_multiagent_session()` encapsulates a full session with turn-level tracking via `begin_turn()` / `end_turn()`.

In [ ]:
def run_multiagent_session(conversation: list, session_label: str) -> MetricsCollector:
    """Run a full conversation session and return the metrics collector."""
    ts = datetime.now().strftime("%Y%m%d%H%M%S")
    sid = f"{session_label}-{ts}"
    # Shared actor_id so all spokes can read each other's writes
    shared_aid = f"travel-team-{ts}"

    collector = MetricsCollector(region=region)
    turn_counter = 0

    @tool
    def flight_booking_assistant(query: str) -> str:
        """Process flight booking queries.
        Args:
            query: A flight-related question
        Returns:
            Flight information and booking options
        """
        collector.record_handoff("flight", query)
        hook = CustomInstrumentationHook(memory_client, memory_id, collector, "flight")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=FLIGHT_PROMPT,
                      state={"actor_id": shared_aid, "session_id": sid})
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("flight", time.perf_counter() - t0)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage("flight",
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    @tool
    def hotel_booking_assistant(query: str) -> str:
        """Process hotel booking queries.
        Args:
            query: A hotel-related question
        Returns:
            Hotel information and booking options
        """
        collector.record_handoff("hotel", query)
        hook = CustomInstrumentationHook(memory_client, memory_id, collector, "hotel")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=HOTEL_PROMPT,
                      state={"actor_id": shared_aid, "session_id": sid})
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("hotel", time.perf_counter() - t0)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage("hotel",
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    @tool
    def itinerary_assistant(query: str) -> str:
        """Build a travel itinerary from flight and hotel results.
        Args:
            query: Request to build an itinerary
        Returns:
            A cohesive travel itinerary
        """
        collector.record_handoff("itinerary", query)
        hook = CustomInstrumentationHook(memory_client, memory_id, collector, "itinerary")
        agent = Agent(hooks=[hook], model=AGENT_MODEL_ID, system_prompt=ITINERARY_PROMPT,
                      state={"actor_id": shared_aid, "session_id": sid})
        t0 = time.perf_counter()
        resp = agent(query)
        collector.record_agent_latency("itinerary", time.perf_counter() - t0)
        usage = getattr(resp, "usage", None) or {}
        collector.record_token_usage("itinerary",
            usage.get("inputTokens", 0), usage.get("outputTokens", 0))
        return str(resp)

    hub = Agent(system_prompt=HUB_PROMPT, model=AGENT_MODEL_ID,
               tools=[flight_booking_assistant, hotel_booking_assistant, itinerary_assistant])

    for msg in conversation:
        turn_counter += 1
        collector.begin_turn(turn_counter, msg)
        print(f"\n{'='*60}")
        print(f"[{session_label}] Turn {turn_counter}: {msg}")
        print(f"{'='*60}")
        resp = hub(msg)
        collector.end_turn()
        print(f"\nHub: {str(resp)[:300]}...")

    return collector

## Step 5 — Simple Session

Fixed budget, no mid-session changes. This is the baseline.

In [ ]:
simple_conversation = [
    "Book a trip from LA to NYC, July 10 to July 17, 1 traveler. "
    "Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.",
    "Now build me a day-by-day itinerary for the trip based on the flight and hotel you found.",
]

simple_conv_metrics = run_multiagent_session(simple_conversation, "simple")

### Simple Session — Context Flow Trace

This shows exactly what each agent received (handoff + memory read) and produced (response) at every turn. Follow the flow to build a mental model of how context moves through the system.

In [ ]:
display(Markdown(simple_conv_metrics.trace_report()))

## Step 6 — Session with Feedback

Same initial request, but the user changes the budget mid-session. This tests:
- Does the coordinator re-dispatch with the updated budget?
- Do the spokes read the latest context from memory?
- Does the itinerary agent produce a plan consistent with the new budget?

In [ ]:
feedback_conversation = [
    "Book a trip from LA to NYC, July 10 to July 17, 1 traveler. "
    "Budget $1800 for flights. I want morning flights and a hotel in Midtown with a pool.",
    "Actually, my budget changed to $1400 total for flights. "
    "Please redo the flight and hotel search with this new budget.",
    "Now build me a day-by-day itinerary for the trip based on the updated flight and hotel.",
]

feedback_metrics = run_multiagent_session(feedback_conversation, "feedback")

### Feedback Session — Context Flow Trace

In [ ]:

display(Markdown(feedback_metrics.trace_report()))

## Step 7 — Run LLM-as-Judge Evaluations

This runs Claude Opus as a judge on every agent call in both sessions. It evaluates context freshness, handoff completeness, utilization, write accuracy, redundancy, and cross-agent consistency.

In [ ]:
print("Evaluating simple session...")
simple_conv_metrics.evaluate_all()

print("Evaluating feedback session...")
feedback_metrics.evaluate_all()

## Step 8 — Results: Simple Session

In [ ]:
display(Markdown("## Simple Session\n\n" + simple_conv_metrics.context_metrics_report()))
display(Markdown(simple_conv_metrics.latency_metrics_report()))

## Step 9 — Results: Session with Feedback

In [ ]:
display(Markdown("## Session with Feedback\n\n" + feedback_metrics.context_metrics_report()))
display(Markdown(feedback_metrics.latency_metrics_report()))

## Step 10 — Side-by-Side Comparison

In [ ]:
display(Markdown(MetricsCollector.comparison_report(
    simple_conv_metrics, feedback_metrics,
    label_a="Simple Session", label_b="Session with Feedback")))

## Visual Analysis

In [ ]:
import matplotlib.pyplot as plt
from eval_helpers import (
    plot_context_metrics_radar, plot_latency_breakdown,
    plot_session_comparison
)

In [ ]:
fig = plot_context_metrics_radar(simple_conv_metrics, "Simple Session")
plt.show()

fig = plot_context_metrics_radar(feedback_metrics, "Feedback Session")
plt.show()

In [ ]:
fig = plot_latency_breakdown(simple_conv_metrics, "Simple Session")
plt.show()

fig = plot_latency_breakdown(feedback_metrics, "Feedback Session")
plt.show()

In [ ]:
fig = plot_session_comparison(simple_conv_metrics, feedback_metrics,
                              "Simple Session", "Feedback Session")
plt.show()

## Clean Up

In [ ]:
# Uncomment to delete the memory resource
# memory_client.delete_memory_and_wait(memory_id=memory_id, max_wait=300, poll_interval=10)